# Evaluation Hybride — N-gram + LM fine-tune (ISO 7240-14)

Compare sur le **meme test set** (seed=42, identique aux notebooks GPT-2/Qwen2) :
- N-gram seul (entraine sur train uniquement)
- LM fine-tune seul (GPT-2 ou Qwen2)
- Hybride avec 4 seuils differents

Le hybride utilise le n-gram quand il est confiant, le LM sinon.


## 0. GPU + install

In [ ]:
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU seulement")

GPU : Tesla T4


In [ ]:
!pip install -q transformers

## 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR   = '/content/drive/MyDrive/autocomplete'
CORPUS_PATH   = f'{PROJECT_DIR}/data/scenarios_fire_detection_clean.txt'
GLOSSARY_PATH = f'{PROJECT_DIR}/data/glossaire_fire_detection_and_alarm_systems.txt'
GPT2_DIR      = f'{PROJECT_DIR}/models/gpt2_fire_detection'
QWEN2_DIR     = f'{PROJECT_DIR}/models/qwen2_fire_detection'

import os
print("Corpus    :", "OK" if os.path.exists(CORPUS_PATH) else "MANQUANT")
print("GPT-2     :", "OK" if os.path.exists(GPT2_DIR)   else "MANQUANT")
print("Qwen2     :", "OK" if os.path.exists(QWEN2_DIR)  else "MANQUANT")

Mounted at /content/drive
Corpus    : OK
GPT-2     : OK
Qwen2     : OK


## 2. Code N-gram, LM, Hybride

In [ ]:
import re, random
from collections import defaultdict, Counter

class NgramAutocomplete:
    def __init__(self, n=3, glossary_path=None):
        self.n = n
        self.ngram_counts = defaultdict(Counter)
        self.word_freq = Counter()
        self.glossary_words = set()
        if glossary_path and os.path.exists(glossary_path):
            with open(glossary_path, encoding='utf-8') as f:
                self.glossary_words = {l.strip().lower() for l in f if l.strip()}

    def tokenize(self, text):
        return re.findall(r'\b[a-zA-Z]+\b', text.lower())

    def train_sentence(self, sentence):
        tokens = self.tokenize(sentence)
        self.word_freq.update(tokens)
        for order in range(2, self.n + 1):
            for i in range(len(tokens) - order):
                ctx  = tuple(tokens[i:i+order-1])
                next_w = tokens[i+order-1]
                self.ngram_counts[ctx][next_w] += 1

    def predict(self, context_words, prefix='', top_k=5):
        results, seen = [], set()
        for order in range(self.n-1, 0, -1):
            if len(context_words) >= order:
                ctx = tuple(context_words[-order:])
                if ctx in self.ngram_counts:
                    total = sum(self.ngram_counts[ctx].values())
                    for word, score in self.ngram_counts[ctx].most_common():
                        if word.startswith(prefix) and word not in seen:
                            results.append((word, round(score/total*100, 1), word in self.glossary_words))
                            seen.add(word)
                    break
        total_freq = sum(self.word_freq.values())
        for word, score in self.word_freq.most_common():
            if len(results) >= top_k: break
            if word.startswith(prefix) and word not in seen:
                results.append((word, round(score/total_freq*100, 2), word in self.glossary_words))
                seen.add(word)
        return results[:top_k]
print("NgramAutocomplete OK")

NgramAutocomplete OK


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

class LMFinetunedAutocomplete:
    def __init__(self, model_name='gpt2', output_dir=''):
        self.tokenizer = AutoTokenizer.from_pretrained(
            output_dir if os.path.exists(output_dir) else model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        src = output_dir if os.path.exists(output_dir) else model_name
        self.model = AutoModelForCausalLM.from_pretrained(src)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device).eval()
        vocab_size = len(self.tokenizer)
        self._tok_alpha = []
        for tid in range(vocab_size):
            s = self.tokenizer.decode([tid]).strip().lower()
            self._tok_alpha.append(s if re.match(r'[a-z]', s) else '')

    def predict(self, context_words, prefix='', top_k=5):
        sentence = ' '.join(context_words).strip() or self.tokenizer.eos_token
        enc = self.tokenizer(sentence, return_tensors='pt').to(self.device)
        with torch.no_grad():
            logits = self.model(**enc).logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)
        order = torch.argsort(probs, descending=True).tolist()
        preds, seen = [], set()
        for tid in order:
            w = self._tok_alpha[tid]
            if not w: continue
            w = re.sub(r'[^a-z]', '', w)
            if not w or w in seen: continue
            if prefix and not w.startswith(prefix): continue
            preds.append((w, round(float(probs[tid])*100, 2), False))
            seen.add(w)
            if len(preds) >= top_k: break
        return preds
print("LMFinetunedAutocomplete OK")

LMFinetunedAutocomplete OK


In [ ]:
class HybridAutocomplete:
    def __init__(self, ngram, lm, confidence_threshold=5.0):
        self.ngram = ngram
        self.lm    = lm
        self.threshold = confidence_threshold

    def predict(self, context_words, prefix='', top_k=5):
        ngram_results = self.ngram.predict(context_words, prefix, top_k=top_k)
        if ngram_results and ngram_results[0][1] >= self.threshold:
            return ngram_results
        lm_results = self.lm.predict(context_words, prefix, top_k=top_k)
        seen = {r[0] for r in lm_results}
        combined = list(lm_results)
        for word, conf, is_g in ngram_results:
            if word not in seen:
                combined.append((word, conf, is_g)); seen.add(word)
        return combined[:top_k]

    def which_model(self, context_words, prefix=''):
        r = self.ngram.predict(context_words, prefix, top_k=1)
        return 'ngram' if (r and r[0][1] >= self.threshold) else 'lm'
print("HybridAutocomplete OK")

HybridAutocomplete OK


## 3. Split 70/15/15 (meme seed=42 que les notebooks)

In [ ]:
with open(CORPUS_PATH, encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]

random.seed(42)
shuffled = lines[:]
random.shuffle(shuffled)
n = len(shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
train_lines = shuffled[:n_train]
val_lines   = shuffled[n_train:n_train+n_val]
test_lines  = shuffled[n_train+n_val:]
print(f"Train={len(train_lines)} | Val={len(val_lines)} | Test={len(test_lines)}")

Train=210 | Val=45 | Test=46


## 4. Evaluation

In [ ]:
PREFIX_LEN = 2

def tokenize(text):
    return re.findall(r'\b[a-zA-Z]+\b', text.lower())

def evaluate(model, test_lines, label='', is_hybrid=False):
    top1, top3, total = 0, 0, 0
    ngram_used = lm_used = 0
    for line in test_lines:
        words = tokenize(line)
        for i in range(1, len(words)):
            target = words[i]
            if len(target) <= PREFIX_LEN: continue
            prefix  = target[:PREFIX_LEN]
            context = words[:i]
            try:
                if is_hybrid:
                    src = model.which_model(context, prefix)
                    if src == 'ngram': ngram_used += 1
                    else:              lm_used    += 1
                results = model.predict(context, prefix, top_k=3)
                ws = [r[0] for r in results]
                if ws and ws[0] == target: top1 += 1
                if target in ws:           top3 += 1
            except: pass
            total += 1
    res = {'label': label,
            'Top-1': round(top1/total*100,1) if total else 0,
            'Top-3': round(top3/total*100,1) if total else 0,
            'tests': total}
    if is_hybrid:
        tot_u = ngram_used + lm_used
        res['ngram_%'] = round(ngram_used/tot_u*100) if tot_u else 0
        res['lm_%']    = round(lm_used/tot_u*100)    if tot_u else 0
    return res

def print_results(results_list):
    print(f"\n{'Modele':<42} {'Top-1':>7} {'Top-3':>7}")
    print('-'*60)
    for r in results_list:
        print(f"  {r['label']:<40} {r['Top-1']:>6}%  {r['Top-3']:>6}%")
    print()
    for r in results_list:
        if 'ngram_%' in r:
            print(f"  {r['label']:<40} N-gram:{r['ngram_%']}%  LM:{r['lm_%']}%")
print("Fonctions evaluation OK")

Fonctions evaluation OK


## 5. Chargement des modeles

In [ ]:
# N-gram entraine sur train uniquement
print("Chargement N-gram...")
ngram = NgramAutocomplete(n=3, glossary_path=GLOSSARY_PATH)
for s in train_lines:
    ngram.train_sentence(s)
print(f"  N-gram entraine sur {len(train_lines)} phrases")

lm, lm_label = None, ''
for model_dir, base, label in [
    (QWEN2_DIR, 'Qwen/Qwen2-0.5B', 'Qwen2-0.5B fine-tune'),
    (GPT2_DIR,  'gpt2',            'GPT-2 fine-tune'),
]:
    if os.path.exists(model_dir):
        print(f"Chargement {label}...")
        lm = LMFinetunedAutocomplete(model_name=base, output_dir=model_dir)
        lm_label = label
        break

if lm is None:
    print("Aucun modele fine-tune trouve.")

Chargement N-gram...
  N-gram entraine sur 210 phrases
Chargement Qwen2-0.5B fine-tune...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 6. Lancer l'evaluation

In [ ]:
results = []

print("Evaluation N-gram...")
results.append(evaluate(ngram, test_lines, label="N-gram (n=3)"))

if lm:
    print(f"Evaluation {lm_label}...")
    results.append(evaluate(lm, test_lines, label=lm_label))

    print("Evaluation hybride (seuils 2, 5, 10, 20)...")
    for threshold in [2.0, 5.0, 10.0, 20.0]:
        hybrid = HybridAutocomplete(ngram=ngram, lm=lm,
                                    confidence_threshold=threshold)
        results.append(evaluate(hybrid, test_lines,
                                label=f"Hybride seuil={threshold}",
                                is_hybrid=True))

print_results(results)

Evaluation N-gram...
Evaluation Qwen2-0.5B fine-tune...
Evaluation hybride (seuils 2, 5, 10, 20)...

Modele                                       Top-1   Top-3
------------------------------------------------------------
  N-gram (n=3)                               61.3%    79.7%
  Qwen2-0.5B fine-tune                       77.3%    86.9%
  Hybride seuil=2.0                          77.5%    88.8%
  Hybride seuil=5.0                          79.9%    90.3%
  Hybride seuil=10.0                         80.5%    90.2%
  Hybride seuil=20.0                         79.9%    89.6%

  Hybride seuil=2.0                        N-gram:55%  LM:45%
  Hybride seuil=5.0                        N-gram:44%  LM:56%
  Hybride seuil=10.0                       N-gram:36%  LM:64%
  Hybride seuil=20.0                       N-gram:31%  LM:69%


In [ ]:
import re, torch

def next_words_hybrid(sentence, k=5, threshold=10.0):
    """Predit les k mots SUIVANTS apres `sentence` complete."""
    hybrid  = HybridAutocomplete(ngram=ngram, lm=lm, confidence_threshold=threshold)
    context = re.findall(r'\b[a-zA-Z]+\b', sentence.lower())

    results = hybrid.predict(context, prefix='', top_k=k)
    source  = hybrid.which_model(context, prefix='')
    return [r[0] for r in results], source


phrase = ""
while True:
    mot = input("mot suivant (vide pour arreter) : ")
    if not mot.strip():
        break
    phrase = (phrase + " " + mot).strip()
    preds, source = next_words_hybrid(phrase)
    print(f"  phrase      : {phrase}")
    print(f"  source      : {source}")
    print(f"  suggestions : {preds}")

mot suivant (vide pour arreter) : the
  phrase      : the
  source      : ngram
  suggestions : ['fdas', 'design', 'fdcie', 'operation', 'designer']
mot suivant (vide pour arreter) : fire
  phrase      : the fire
  source      : ngram
  suggestions : ['alarm', 'detection', 'the', 'of', 'shall']
mot suivant (vide pour arreter) : detection
  phrase      : the fire detection
  source      : ngram
  suggestions : ['and', 'system', 'control', 'the', 'of']
mot suivant (vide pour arreter) : systeù
  phrase      : the fire detection systeù
  source      : ngram
  suggestions : ['and', 'system', 'control', 'the', 'of']
mot suivant (vide pour arreter) : 
